# 02 — Buffering Lines

A point buffer answers: *what is within X km of this location?*

A line buffer answers: *what is within X km of anywhere along this path?*

The result is a **corridor** — a zone of influence that follows the path, with rounded caps at each end. For a missile trajectory, this corridor represents the airspace that could be affected by proximity to the flight path.

## Setup

In [1]:
import json
import math
from pathlib import Path
from ipyleaflet import Map, GeoJSON, basemaps

DATA_DIR = Path("data")

with open(DATA_DIR / "targets.geojson") as f:
    targets_fc = json.load(f)

# Load paths from module 09 if available, otherwise define inline
paths_file = Path("../09-Intersections/data/paths.geojson")
if paths_file.exists():
    with open(paths_file) as f:
        module_paths = json.load(f)
else:
    module_paths = {
        "type": "FeatureCollection",
        "features": [
            {"type": "Feature", "properties": {"name": "Alpha"},
             "geometry": {"type": "LineString",
                          "coordinates": [[-77.009, 38.889], [51.388, 35.695]]}},
            {"type": "Feature", "properties": {"name": "Delta"},
             "geometry": {"type": "LineString",
                          "coordinates": [[37.617, 55.755], [46.675, 24.688]]}},
        ]
    }


# Buffer helpers from notebook 01
def destination_point(lon, lat, bearing_deg, distance_km):
    R = 6371.0
    d = distance_km / R
    phi1 = math.radians(lat)
    lam1 = math.radians(lon)
    theta = math.radians(bearing_deg)
    phi2 = math.asin(math.sin(phi1)*math.cos(d) + math.cos(phi1)*math.sin(d)*math.cos(theta))
    lam2 = lam1 + math.atan2(math.sin(theta)*math.sin(d)*math.cos(phi1),
                              math.cos(d) - math.sin(phi1)*math.sin(phi2))
    return [math.degrees(lam2), math.degrees(phi2)]

def point_buffer(lon, lat, radius_km, n_points=48):
    ring = [destination_point(lon, lat, 360.0*i/n_points, radius_km) for i in range(n_points)]
    ring.append(ring[0])
    return {"type": "Polygon", "coordinates": [ring]}

print("Ready.")

Ready.


## Why Not Just a Rectangle?

The simplest model of a line buffer would be a rectangle — offset the line by the radius on each side and cap the ends. That works for flat-plane geometry, but on Earth's surface:

- The path curves on a sphere
- The perpendicular direction changes at every point along the path
- The end caps should be semicircles, not flat edges

The **sampling approach** sidesteps all of this cleanly: place a circular buffer at each sampled point along the path. With enough samples they overlap continuously, producing a smooth corridor with naturally rounded ends.

## Sampling Points Along a Segment

For a two-point segment, linear interpolation in lon/lat gives evenly-spaced samples. For a multi-leg path, we sample proportionally across all segments.

In [2]:
def sample_linestring(coords, n_samples):
    """
    Sample n_samples points evenly along a LineString defined by coords.
    Uses linear interpolation within each segment.
    """
    if len(coords) < 2:
        return coords[:]

    n_segs = len(coords) - 1
    per_seg = max(1, n_samples // n_segs)
    points = []

    for i in range(n_segs):
        p1, p2 = coords[i], coords[i + 1]
        for j in range(per_seg):
            t = j / per_seg
            points.append([
                p1[0] + t * (p2[0] - p1[0]),
                p1[1] + t * (p2[1] - p1[1])
            ])

    points.append(coords[-1])   # always include the endpoint
    return points


# Test on Alpha path (D.C. → Tehran)
alpha = next(f for f in module_paths["features"] if f["properties"]["name"] == "Alpha")
samples = sample_linestring(alpha["geometry"]["coordinates"], n_samples=10)
print(f"10 samples along Alpha path:")
for i, pt in enumerate(samples):
    print(f"  {i:2d}  lon={pt[0]:8.3f}  lat={pt[1]:7.3f}")

10 samples along Alpha path:
   0  lon= -77.009  lat= 38.889
   1  lon= -64.169  lat= 38.570
   2  lon= -51.330  lat= 38.250
   3  lon= -38.490  lat= 37.931
   4  lon= -25.650  lat= 37.611
   5  lon= -12.811  lat= 37.292
   6  lon=   0.029  lat= 36.973
   7  lon=  12.869  lat= 36.653
   8  lon=  25.709  lat= 36.334
   9  lon=  38.548  lat= 36.014
  10  lon=  51.388  lat= 35.695


## Building the Corridor

Place a `point_buffer` at every sampled position. Together they form the corridor. The key parameters:

- `radius_km` — half-width of the corridor
- `n_samples` — how many circles to place (more = smoother, slower)
- `n_points` — vertices per circle (32 is fine for corridors)

In [3]:
def line_buffer(path_feature, radius_km, n_samples=40, n_points=32):
    """
    Return a FeatureCollection of overlapping circular buffers along a
    LineString path — together they form a corridor zone.
    """
    coords  = path_feature["geometry"]["coordinates"]
    samples = sample_linestring(coords, n_samples)
    name    = path_feature["properties"].get("name", "")

    return {
        "type": "FeatureCollection",
        "features": [
            {
                "type": "Feature",
                "properties": {"path": name, "radius_km": radius_km},
                "geometry": point_buffer(pt[0], pt[1], radius_km, n_points)
            }
            for pt in samples
        ]
    }


print(f"line_buffer defined — takes a path feature, returns a corridor FeatureCollection")

line_buffer defined — takes a path feature, returns a corridor FeatureCollection


## Visualizing a Path Corridor

In [4]:
# Corridor around the Alpha path (D.C. → Tehran), 200 km wide
corridor = line_buffer(alpha, radius_km=200, n_samples=50)

print(f"Corridor: {len(corridor['features'])} circle polygons")

m = Map(center=(40, 10), zoom=3, basemap=basemaps.CartoDB.Positron)

# Draw corridor first (bottom layer)
m.add(GeoJSON(
    data=corridor,
    style={"color": "#2980b9", "fillColor": "#2980b9", "fillOpacity": 0.15, "weight": 0}
))
# Draw path on top
m.add(GeoJSON(
    data={"type": "FeatureCollection", "features": [alpha]},
    style={"color": "#2980b9", "weight": 2, "opacity": 0.9}
))
m

Corridor: 51 circle polygons


Map(center=[40, 10], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_tex…

## End Caps

Because each sample point gets a full circular buffer, the endpoints automatically get semicircular caps — the portion of the endpoint circles that extends beyond the line. No special handling required.

The quality of the cap depends on `n_samples`: with too few samples, the endpoint circle may not overlap the adjacent circles enough, leaving a visual gap. A rule of thumb: make sure the sample interval is less than `radius_km`.

In [5]:
# Show end-cap detail: zoom in on the Washington D.C. end of Alpha
end_cap_samples = sample_linestring(alpha["geometry"]["coordinates"], 10)
end_cap_buffers = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature", "properties": {},
            "geometry": point_buffer(pt[0], pt[1], 200, 32)
        }
        for pt in end_cap_samples[:4]   # first 4 samples only
    ]
}

m_cap = Map(center=(38.9, -77), zoom=5, basemap=basemaps.CartoDB.Positron)
m_cap.add(GeoJSON(
    data=end_cap_buffers,
    style={"color": "#2980b9", "fillColor": "#2980b9", "fillOpacity": 0.2, "weight": 1}
))
m_cap

Map(center=[38.9, -77], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_…

## Exercise A

Buffer the **Delta** path (Moscow → Riyadh) with a `radius_km` of 150 km and `n_samples=40`.

1. Use `line_buffer` to create the corridor.
2. Display the corridor and the path line on the same map.
3. Print the number of circle polygons in the corridor.

In [6]:
from ipyleaflet import Map, GeoJSON, basemaps

# Get the Delta path
delta = next(
    f for f in module_paths["features"]
    if f["properties"]["name"] == "Delta"
)

# 1. Create 150 km corridor with 40 samples
delta_corridor = line_buffer(
    delta,
    radius_km=150,
    n_samples=40
)

# 3. Print circle count
print(f"Delta corridor circle polygons: {len(delta_corridor['features'])}")

# 2. Display corridor + path
m = Map(center=(40, 45), zoom=4, basemap=basemaps.CartoDB.Positron)

# Corridor layer
m.add(GeoJSON(
    data=delta_corridor,
    name="Delta 150 km corridor",
    style={
        "color": "#8e44ad",
        "fillColor": "#8e44ad",
        "fillOpacity": 0.15,
        "weight": 0
    }
))

# Path line layer
m.add(GeoJSON(
    data={
        "type": "FeatureCollection",
        "features": [delta]
    },
    name="Delta path",
    style={
        "color": "#e74c3c",
        "weight": 3,
        "opacity": 0.9
    }
))

m

Delta corridor circle polygons: 41


Map(center=[40, 45], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_tex…

## Exercise B

Compare a **point buffer** and a **line buffer** for the same feature and radius.

1. Take the endpoint of the Alpha path (Tehran, `[51.388, 35.695]`).
2. Create a 200 km **point buffer** around Tehran.
3. Create a 200 km **line buffer** for the full Alpha path.
4. Display both on the same map. Describe the difference: which covers more area, and why?

In [7]:
from ipyleaflet import Map, GeoJSON, basemaps

# Get Alpha path
alpha = next(
    f for f in module_paths["features"]
    if f["properties"]["name"] == "Alpha"
)

# 1-2. Point buffer around Tehran endpoint
tehran = [51.388, 35.695]

tehran_buffer = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "properties": {"name": "Tehran 200 km point buffer"},
            "geometry": point_buffer(tehran[0], tehran[1], 200)
        }
    ]
}

# 3. Line buffer for full Alpha path
alpha_line_buffer = line_buffer(
    alpha,
    radius_km=200,
    n_samples=50
)

# Alpha path layer
alpha_path_fc = {
    "type": "FeatureCollection",
    "features": [alpha]
}

# Tehran point layer
tehran_point_fc = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "properties": {"name": "Tehran"},
            "geometry": {
                "type": "Point",
                "coordinates": tehran
            }
        }
    ]
}

# 4. Display both on same map
m = Map(center=(40, 10), zoom=3, basemap=basemaps.CartoDB.Positron)

# Line buffer first
m.add(GeoJSON(
    data=alpha_line_buffer,
    name="Alpha 200 km line buffer",
    style={
        "color": "#2980b9",
        "fillColor": "#2980b9",
        "fillOpacity": 0.15,
        "weight": 0
    }
))

# Point buffer on top
m.add(GeoJSON(
    data=tehran_buffer,
    name="Tehran 200 km point buffer",
    style={
        "color": "#e74c3c",
        "fillColor": "#e74c3c",
        "fillOpacity": 0.25,
        "weight": 2
    }
))

# Alpha path line
m.add(GeoJSON(
    data=alpha_path_fc,
    name="Alpha path",
    style={
        "color": "#2c3e50",
        "weight": 3,
        "opacity": 0.9
    }
))

# Tehran point
m.add(GeoJSON(
    data=tehran_point_fc,
    name="Tehran point"
))

m

Map(center=[40, 10], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_tex…

---

## Check Your Understanding

**1.** Why is a line buffer not just a rectangle, even for a straight-line path on Earth?

**2.** What happens at the **ends** of a buffered line, and what controls the quality of those end caps?

```python
# No code needed — answer in your own words
```

## Check Your Understanding

### 1.
Why is a line buffer not just a rectangle, even for a straight-line path on Earth?

A line buffer is not just a rectangle because the buffer must include all points within a certain distance of the line, including around the ends.

This creates rounded edges and curved boundaries instead of sharp rectangular corners.

Also, Earth is curved, so distances and directions change across geographic coordinates. A true geographic buffer follows spherical geometry rather than simple flat geometry.

In this notebook, the corridor is approximated using many overlapping circular buffers, which naturally produces a rounded corridor shape instead of a rectangle.

---

### 2.
What happens at the ends of a buffered line, and what controls the quality of those end caps?

At the ends of the line, the buffer forms rounded “end caps” that extend around the start and end points of the path.

These end caps are created by the circular buffers placed at the endpoints.

The quality and smoothness of the end caps are controlled mainly by:
- `n_points` → how smooth each circle polygon looks
- `n_samples` → how densely circles are placed along the line

Higher values create smoother, more continuous-looking ends and corridors.

## Next

In [03 — Comparing Buffer Sizes](./03-Comparing_Buffer_Sizes.ipynb), we stack multiple radii on the same feature to model intensity zones — lethal, damaging, and warning rings around a single target.